# First baseline experiment

This notebook prepares a majority-class reference and a logistic-regression model.
You control installation, preparation, and training separately. All three switches default to false.
Exporting this notebook as a report does not start training with these defaults.
Use a CPU runtime. These models do not require a GPU.


In [ ]:
import json
import logging
import os
import shlex
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython import get_ipython
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("baseline_notebook")

RUN_SETUP = False
RUN_PREPARATION = False
RUN_TRAINING = False

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "main"
local_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/baseline.yaml").exists()), None
)
PROJECT_DIR = local_root or Path("/content/sleep-classifier")
project_argument = shlex.quote(str(PROJECT_DIR))
logger.info("Project directory: %s", PROJECT_DIR)

## 1. Open the checkout and install uv

For a fresh Colab runtime, enable RUN_SETUP first.
The installer uses Astral's official installation script. Existing checkouts remain unchanged.
Select the experiment branch before its merge. Use main after the experiment branch merges.


In [ ]:
if RUN_SETUP:
    if not PROJECT_DIR.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} {project_argument}
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        # Download the official installer before executing it so curl failures stop setup
        installer_path = PROJECT_DIR / "artifacts/uv-install.sh"
        installer_path.parent.mkdir(parents=True, exist_ok=True)
        installer_argument = shlex.quote(str(installer_path))
        !curl -LsSf https://astral.sh/uv/install.sh -o {installer_argument}
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installer download failed")
        !sh {installer_argument} --no-modify-path
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    !uv sync --project {project_argument} --locked --group baseline --group notebook
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Dependency installation failed")
    !git -C {project_argument} rev-parse HEAD
else:
    logger.info("Setup skipped")

## 2. Prepare data

Enable RUN_PREPARATION on a fresh checkout. The archive requires about 550 MB of network transfer.
Integrity checks include all participants. Feature preparation opens training and validation observations only.
Preparation uses the saved split and coverage rule. It does not train a model.


In [ ]:
if RUN_PREPARATION:
    %cd "{PROJECT_DIR}"
    !uv run --locked --group notebook --group baseline python -m sleep_classifier.download_cohort \
        --archive data/sleep-accel-1.0.0.zip --destination data/raw/sleep-accel/1.0.0
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Cohort download failed")
    !uv run --locked --group notebook --group baseline python -m \
        sleep_classifier.cohort inventory \
        --config configs/cohort.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Cohort inventory failed")
    !uv run --locked --group notebook --group baseline python -m \
        sleep_classifier.cohort split \
        --config configs/cohort.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Participant split verification failed")
    !uv run --locked --group notebook --group baseline python -m \
        sleep_classifier.prepare \
        --config configs/prepare.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Feature preparation failed")
else:
    logger.info("Data preparation skipped")

## 3. Train when ready

Enable RUN_TRAINING to fit both baselines. Review configs/baseline.yaml before running this cell.
The scaler fits eligible training rows only. Validation rows receive the fitted transformation.
The classifiers use the six explicit sensor features. Labels and coverage fields remain excluded from model inputs.

Equivalent command from the checkout:
```bash
uv run --locked --group baseline python -m sleep_classifier.baseline --config configs/baseline.yaml --train
```

Logistic regression has optimizer iterations, not training epochs. Logs show stages, elapsed times, and final iteration counts.


In [ ]:
if RUN_TRAINING:
    %cd "{PROJECT_DIR}"
    !uv run --locked --group notebook --group baseline python -u -m sleep_classifier.baseline \
        --config configs/baseline.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Baseline training failed")
else:
    logger.info("Training skipped. Enable RUN_TRAINING when ready to fit the models.")

## 4. Review saved validation results

Metrics describe retained validation epochs. Coverage uses all scored validation epochs as its denominator.
Compare accuracy, macro-F1, REM precision/recall/F1, and individual participants.
The final test set remains reserved for the fixed model protocol.
Old metrics remain available until you replace them with a new training run.
Inspect the saved configuration and provenance before comparing runs.


In [ ]:
metrics_path = PROJECT_DIR / "artifacts/baseline/metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    logger.info("Saved configuration: %s", metrics["config"])
    display(
        pd.DataFrame(
            [
                {
                    "model": name,
                    "accuracy": model["accuracy"],
                    "macro_f1": model["macro_f1"],
                    "coverage": model["coverage"]["fraction"],
                    "REM_recall": model["classes"]["3"]["recall"],
                }
                for name, model in metrics["models"].items()
            ]
        ).set_index("model")
    )
    class_names = ["wake", "light", "deep", "REM"]
    for name, model in metrics["models"].items():
        logger.info("%s: class metrics and confusion matrix (rows=true, columns=predicted)", name)
        display(pd.DataFrame(model["classes"]).T.rename(index=dict(zip("0123", class_names))))
        display(pd.DataFrame(model["confusion_matrix"], index=class_names, columns=class_names))
else:
    logger.info("No saved baseline results. Run training when ready.")

In [ ]:
if metrics_path.exists():
    import matplotlib.pyplot as plt
    import numpy as np

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, (name, model) in zip(axes, metrics["models"].items()):
        counts = np.asarray(model["confusion_matrix"])
        row_fractions = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        image = axis.imshow(row_fractions, vmin=0, vmax=1, cmap="Blues")
        axis.set_xticks(range(4), class_names)
        axis.set_yticks(range(4), class_names)
        axis.set(xlabel="Predicted stage", ylabel="True stage", title=name)
        for row in range(4):
            for column in range(4):
                axis.text(
                    column,
                    row,
                    f"{row_fractions[row, column]:.1%}",
                    ha="center",
                    va="center",
                    color="white" if row_fractions[row, column] > 0.5 else "black",
                )
    figure.colorbar(image, ax=axes, label="Fraction within true stage", shrink=0.8)
    plt.show()

## 5. Export the report

Run this command after training. The report runner skips the tagged training cell, including when RUN_TRAINING is enabled interactively.
The report reads the saved metrics and writes HTML only.

```bash
uv run --locked --group notebook --group baseline python scripts/execute_notebook.py --notebook notebooks/05_baseline_experiment.ipynb
```

Keep the source notebook switches false before committing or exporting it.
The HTML report is written to artifacts/notebooks/05_baseline_experiment.executed.html.

The export cell is also skipped during report execution to prevent recursive report generation.


In [ ]:
%cd "{PROJECT_DIR}"
!uv run --locked --group notebook --group baseline python scripts/execute_notebook.py \
    --notebook notebooks/05_baseline_experiment.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("HTML report export failed")